In [1]:
import numpy as np
import sglang as sgl
from token2action import TokenToAction, image_qa
import pandas as pd
import json
import os

/root/miniconda3/envs/sglang/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
converter = TokenToAction()

def batch(batch_size, temp):
    arguments = [
        {
            "image_path": "images/robot.jpg",
            "question": "In: What action should the robot take to {<INSTRUCTION>}?\nOut:",
        }
    ] * batch_size
    states = image_qa.run_batch(
        arguments,
        max_new_tokens=7,
        temperature=temp
    )
    return [converter.convert(s.get_meta_info("action")["output_ids"]).tolist() for s in states]

In [3]:
def run_experiment(quantization=None, batch_size=[], temperature=0):
    # quantization choices=["awq","fp8","gptq","marlin","gptq_marlin","awq_marlin","squeezellm","bitsandbytes",],
    if quantization == "int4":
        quant = "awq"
    elif quantization == "fp16":
        quant = None
    else:
        quant = quantization
    runtime = sgl.Runtime(
        model_path="openvla/openvla-7b",
        tokenizer_path="openvla/openvla-7b",
        disable_cuda_graph=True,
        disable_radix_cache=True,
        chunked_prefill_size=-1,
        quantization = quant
    )
    sgl.set_default_backend(runtime)
    print(f"=== Quantization: {quantization}, Temperature: {temperature} ===")
    result = {
        "quantization": quantization,
        "temperature": temperature,
        "data": {}
    }
    for batch_size in batch_size:
        print(f" running batch size {batch_size}")
        actions = batch(batch_size=batch_size, temp=temperature)
        result["data"][batch_size] = actions
    runtime.shutdown()
    return result

In [6]:
for quantization in ["fp16", "fp8"]:
    for temp in [2]:
        result = run_experiment(quantization=quantization, batch_size=range(50,201,50), temperature=temp)
        with open(f"logs/batch_{quantization}_{temp}.json", "w") as json_file:
            if not os.path.exists("logs"):
                os.makedirs("logs")
            json.dump(result, json_file, indent=4)

Process Process-3:1:
Traceback (most recent call last):
  File "/root/miniconda3/envs/sglang/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/root/miniconda3/envs/sglang/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/root/github/sglang/python/sglang/srt/managers/controller_single.py", line 149, in start_controller_process
    controller = ControllerSingle(
  File "/root/github/sglang/python/sglang/srt/managers/controller_single.py", line 83, in __init__
    self.tp_server = ModelTpServer(
  File "/root/github/sglang/python/sglang/srt/managers/tp_worker.py", line 97, in __init__
    self.model_runner = ModelRunner(
  File "/root/github/sglang/python/sglang/srt/model_executor/model_runner.py", line 132, in __init__
    self.load_model()
  File "/root/github/sglang/python/sglang/srt/model_executor/model_runner.py", line 160, in load_model
    self.vllm_model_config = VllmModelCo

RuntimeError: Initialization failed. Please see the error messages above.